# Model performance monitoring with the ML App client

This notebook runs the latest published **Estates Sell Prices - AutoFEML - Monitoring** pipeline. It reuses the latest immutable prediction dataset already created by batch scoring in the application—predictions are **not uploaded again**. Only the provided actuals file is uploaded as a new immutable version of the existing monitoring-actuals dataset family.

In [ ]:
BUSINESS_CASE_NAME = "Estates Sell Prices"
ACTUALS_DATASET_NAME = "Estates Sell Prices - Scoring Actuals"
PIPELINE_NAME = "Estates Sell Prices - AutoFEML - Monitoring"
ACTUALS_FILENAME = "estates-sale-prices-batch-scoring-100k-actuals.parquet"

## Client and authentication

Install the client from the repository root with `pip install -e .`. In automation, provide `ML_APP_API_URL` and `ML_APP_ACCESS_TOKEN` through a secret manager.

In [ ]:
import getpass
import html
import json
import os
from pathlib import Path

from IPython.display import HTML, display
from ml_app_client import MLAppClient

client = MLAppClient.from_env()
if not os.getenv("ML_APP_ACCESS_TOKEN", "").strip():
    client.login(input("Login: "), getpass.getpass("Password: "))

## Locate the actuals file

Path resolution supports running Jupyter from either the repository root or `examples/API-usage`.

In [ ]:
candidates = [
    Path.cwd() / "examples" / "data" / ACTUALS_FILENAME,
    Path.cwd().parent / "data" / ACTUALS_FILENAME,
]
actuals_file = next((path for path in candidates if path.is_file()), None)
if actuals_file is None:
    raise FileNotFoundError(
        f"Could not find {ACTUALS_FILENAME}; run Jupyter from the repository root or examples/API-usage"
    )
actuals_file

## Upload actuals only

The client streams the Parquet file into the existing `monitoring_actuals` dataset family. The prediction dataset remains the immutable application artifact produced by the earlier scoring run.

In [ ]:
uploaded_actuals = client.upload_dataset_version(
    actuals_file,
    business_case_name=BUSINESS_CASE_NAME,
    dataset_name=ACTUALS_DATASET_NAME,
    description="Observed sale prices for the 100k estates scoring batch",
    tags=["monitoring", "actuals", "estates"],
)
rows = "unknown" if uploaded_actuals.row_count is None else f"{uploaded_actuals.row_count:,}"
print(
    f"Uploaded actuals v{uploaded_actuals.version_number}; "
    f"dataset_id={uploaded_actuals.id}; rows={rows}"
)

## Run monitoring

The latest published monitoring pipeline resolves its prediction input to the latest successful scoring output already stored in the application and its actuals input to the version uploaded above. The full join and all primary metrics run on the complete 100k-row scope.

In [ ]:
run = client.run_pipeline_by_name(
    business_case_name=BUSINESS_CASE_NAME,
    pipeline_name=PIPELINE_NAME,
)
print(f"Started monitoring run_id={run.id}; status={run.status}")

finished = client.wait_for_pipeline_run(
    run,
    timeout=3600,
    on_update=lambda current: print(
        f"status={current.status}; processed_rows={current.processed_row_count}"
    ),
)
print(f"Completed monitoring run {finished.id}")

## Verify the immutable inputs selected by the backend

The run metadata records the exact prediction and actuals dataset IDs and row counts. This makes the monitoring result reproducible even though the pipeline configuration uses `latest`.

In [ ]:
resolved_inputs = finished.raw.get("runtime_parameters", {}).get("resolved_input_versions", {})
for binding, resolved in resolved_inputs.items():
    print(
        f"{binding}: {resolved['dataset_name']} v{resolved['version_number']} "
        f"(dataset_id={resolved['dataset_id']})"
    )

actuals_binding = resolved_inputs.get("process_join_1:actuals", {})
assert actuals_binding.get("dataset_id") == uploaded_actuals.id

## Retrieve the joined result and report

The official run creates two immutable outputs: a full Parquet dataset containing `property_id`, prediction, and actual value, plus a bounded report artifact containing full-scope metrics and limited chart data.

In [ ]:
joined_dataset_id = client.output_dataset_id(finished)
report = client.scoring_report_for_run(
    finished, business_case_name=BUSINESS_CASE_NAME
)
evaluation = report["evaluation"]
print({
    "joined_dataset_id": joined_dataset_id,
    "report_id": report["id"],
    "evaluated_rows": report["evaluated_row_count"],
})

## Build an attractive standalone report

The renderer below uses only Python and inline HTML/SVG, so it adds no plotting dependency. Metric cards and diagnostics represent the full evaluated scope. The residual histogram uses full-data aggregates, while the actual-vs-predicted scatter is the deterministic bounded rendering sample declared by the report contract.

In [ ]:
def _number(value, digits=2):
    return f"{float(value):,.{digits}f}"


def _metric_value(metric):
    value = float(metric.get("value", 0))
    if metric.get("unit") == "ratio" and metric.get("id") != "r2":
        return f"{value:.2%}"
    if metric.get("id") == "r2":
        return f"{value:.4f}"
    return _number(value)


def _scatter_svg(points, width=680, height=300):
    if not points:
        return "<p class='muted'>No scatter points available.</p>"
    actual = [float(item["actual"]) for item in points]
    predicted = [float(item["predicted"]) for item in points]
    low, high = min(actual + predicted), max(actual + predicted)
    span = high - low or 1.0
    x = lambda value: 38 + (value - low) / span * (width - 60)
    y = lambda value: height - 28 - (value - low) / span * (height - 50)
    dots = "".join(
        f"<circle cx='{x(a):.1f}' cy='{y(p):.1f}' r='2.8'/>"
        for a, p in zip(actual, predicted)
    )
    return (
        f"<svg viewBox='0 0 {width} {height}' role='img' aria-label='Actual versus predicted'>"
        f"<line class='ideal' x1='{x(low):.1f}' y1='{y(low):.1f}' x2='{x(high):.1f}' y2='{y(high):.1f}'/>"
        f"<g class='points'>{dots}</g></svg>"
    )


def _qq_svg(points, width=680, height=300):
    if not points:
        return "<p class='muted'>No QQ-plot points available.</p>"
    theoretical = [float(item["theoretical"]) for item in points]
    observed = [float(item["observed"]) for item in points]
    x_low, x_high = min(theoretical), max(theoretical)
    y_low, y_high = min(observed), max(observed)
    x_span, y_span = x_high - x_low or 1.0, y_high - y_low or 1.0
    x = lambda value: 38 + (value - x_low) / x_span * (width - 60)
    y = lambda value: height - 28 - (value - y_low) / y_span * (height - 50)
    x_mean = sum(theoretical) / len(theoretical)
    y_mean = sum(observed) / len(observed)
    denominator = sum((value - x_mean) ** 2 for value in theoretical) or 1.0
    slope = sum(
        (tx - x_mean) * (oy - y_mean)
        for tx, oy in zip(theoretical, observed)
    ) / denominator
    intercept = y_mean - slope * x_mean
    dots = "".join(
        f"<circle cx='{x(tx):.1f}' cy='{y(oy):.1f}' r='3'/>"
        for tx, oy in zip(theoretical, observed)
    )
    return (
        f"<svg viewBox='0 0 {width} {height}' role='img' aria-label='Residual normal QQ plot'>"
        f"<line class='ideal' x1='{x(x_low):.1f}' y1='{y(intercept + slope * x_low):.1f}' "
        f"x2='{x(x_high):.1f}' y2='{y(intercept + slope * x_high):.1f}'/>"
        f"<g class='points'>{dots}</g></svg>"
    )


def _histogram_html(bins):
    maximum = max((int(item["count"]) for item in bins), default=1) or 1
    bars = "".join(
        f"<div class='bar' title='{_number(item['lower'])} to {_number(item['upper'])}: {item['count']:,}' "
        f"style='height:{max(2, 100 * int(item['count']) / maximum):.1f}%'></div>"
        for item in bins
    )
    return f"<div class='histogram'>{bars}</div>"


def render_monitoring_report(report):
    evaluation = report["evaluation"]
    scope = evaluation.get("data_scope", {})
    diagnostics = evaluation.get("join_diagnostics", {})
    residuals = evaluation.get("residuals", {})
    metric_cards = "".join(
        f"<article class='metric'><span>{html.escape(str(item['label']))}</span>"
        f"<strong>{_metric_value(item)}</strong><small>{html.escape(str(item.get('direction', '')))}</small></article>"
        for item in evaluation.get("metrics", [])
    )
    warnings = "".join(
        f"<li>{html.escape(str(item))}</li>" for item in evaluation.get("warnings", [])
    ) or "<li>No report warnings.</li>"
    return f"""<!doctype html><html><head><meta charset='utf-8'><style>
      :root{{--ink:#172033;--muted:#667085;--line:#e4e7ec;--accent:#6d5dfc;--teal:#12a594;}}
      body{{margin:0;background:#f5f7fb;color:var(--ink);font:15px Inter,Segoe UI,sans-serif}}
      .report{{max-width:1080px;margin:24px auto;padding:30px;background:white;border:1px solid var(--line);border-radius:22px;box-shadow:0 18px 50px #22305b18}}
      .eyebrow{{color:var(--accent);font-weight:800;letter-spacing:.12em;text-transform:uppercase;font-size:12px}}
      h1{{margin:7px 0 4px;font-size:30px}} h2{{margin:28px 0 12px;font-size:18px}} .muted,small{{color:var(--muted)}}
      .scope,.metrics{{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin-top:18px}}
      .scope div,.metric{{padding:16px;border:1px solid var(--line);border-radius:14px;background:linear-gradient(145deg,#fff,#fafbff)}}
      .scope strong,.metric strong{{display:block;margin-top:7px;font-size:24px}} .metric small{{display:block;margin-top:4px}}
      .charts{{display:grid;grid-template-columns:1.35fr 1fr;gap:16px}} .panel{{padding:18px;border:1px solid var(--line);border-radius:16px}} .panel-wide{{grid-column:1/-1}}
      svg{{width:100%;background:#fbfcff;border-radius:10px}} .points circle{{fill:var(--accent);opacity:.48}} .ideal{{stroke:var(--teal);stroke-width:2;stroke-dasharray:7 5}}
      .histogram{{height:260px;display:flex;align-items:flex-end;gap:3px;padding:12px;background:#fbfcff;border-radius:10px}}
      .bar{{flex:1;min-height:2px;background:linear-gradient(#8c7fff,var(--accent));border-radius:4px 4px 1px 1px}}
      .warnings{{padding:14px 18px;background:#fffaeb;border:1px solid #fedf89;border-radius:12px}}
      @media(max-width:800px){{.charts{{grid-template-columns:1fr}}}}
    </style></head><body><main class='report'>
      <div class='eyebrow'>ML App · Full-scope monitoring</div>
      <h1>{html.escape(str(report['name']))}</h1>
      <div class='muted'>Report v{report['version_number']} · run {html.escape(str(report['pipeline_run_id']))}</div>
      <section class='scope'>
        <div><span>Scanned rows</span><strong>{int(scope.get('scanned_row_count', 0)):,}</strong></div>
        <div><span>Evaluated rows</span><strong>{int(scope.get('evaluated_row_count', 0)):,}</strong></div>
        <div><span>Missing actuals</span><strong>{int(diagnostics.get('missing_target_row_count', 0)):,}</strong></div>
        <div><span>Coverage</span><strong>{int(scope.get('evaluated_row_count', 0)) / max(1, int(scope.get('scanned_row_count', 0))):.1%}</strong></div>
      </section>
      <h2>Performance metrics</h2><section class='metrics'>{metric_cards}</section>
      <h2>Diagnostics</h2><section class='charts'>
        <article class='panel'><h3>Actual vs predicted</h3>{_scatter_svg(residuals.get('actual_vs_predicted', {}).get('points', []))}<small>Teal line indicates an ideal prediction.</small></article>
        <article class='panel'><h3>Residual distribution</h3>{_histogram_html(residuals.get('histogram', []))}<small>Aggregated over the full evaluated dataset.</small></article>
        <article class='panel panel-wide'><h3>Normal QQ plot of residuals</h3>{_qq_svg(residuals.get('qq_plot', {}).get('points', []))}<small>99 exact residual quantiles computed from the full evaluated dataset; the teal line is the fitted normal-reference line.</small></article>
      </section>
      <h2>Report notes</h2><ul class='warnings'>{warnings}</ul>
    </main></body></html>"""


report_html = render_monitoring_report(report)
display(HTML(report_html))

## Save all outputs

The joined Parquet file is streamed in chunks. The complete bounded report contract is stored as JSON for downstream automation and as a standalone HTML file for sharing or opening in a browser.

In [ ]:
output_directory = Path.cwd() / "monitoring-results" / finished.id
output_directory.mkdir(parents=True, exist_ok=True)

joined_file = client.download_dataset(
    joined_dataset_id, output_directory / "estates-predictions-with-actuals.parquet"
)
report_json_file = output_directory / "monitoring-report.json"
report_json_file.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
report_html_file = output_directory / "monitoring-report.html"
report_html_file.write_text(report_html, encoding="utf-8")

print("Saved outputs:")
for path in [joined_file, report_json_file, report_html_file]:
    print(f"- {path} ({path.stat().st_size:,} bytes)")